In [ ]:
import pandas as pd
import numpy as np
import time

from pathlib import Path
from sklearn.model_selection import cross_val_score
from IPython.display import display, HTML
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

In [ ]:
# Daten laden
DATA_FILE = Path("DATA") / "data_clean.csv"
df = pd.read_csv(DATA_FILE)

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Spaltendefinitionen
question_cols = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
numeric_cols = question_cols + ['age']
categorical_cols = ['gender', 'hand']


# Einheitlicher Preprocessor
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [ ]:
# 1. Pipeline für Logistic Regression
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

pipe_logreg

In [ ]:
# 2. Pipeline für KNN
pipe_knn = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier()),
    ]
)

In [ ]:
# 3. Pipeline für Random Forest
pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

In [ ]:
# 4. Pipeline für Gradient Boosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

In [ ]:
# Dictionary für GridSearch-Tuning
models_config = {
    "LogisticRegression": {
        "pipeline": pipe_logreg,
        "params": {"classifier__C": [0.01, 0.1, 1.0, 10.0]}
    },
    "KNN": {
        "pipeline": pipe_knn,
        "params": {"classifier__n_neighbors": [3, 5, 7, 9], "classifier__weights": ["uniform", "distance"]}
    },
    "RandomForest": {
        "pipeline": pipe_rf,
        "params": {"classifier__n_estimators": [50, 100, 200], "classifier__max_depth": [None, 10, 20]}
    },
    "GradientBoosting": {
        "pipeline": pipe_hgb,
        "params": {"classifier__learning_rate": [0.01, 0.1, 0.2], "classifier__max_iter": [100, 200]}
    }
}

In [ ]:
best_models = {}
results_html = ""

for name, config in models_config.items():
    # Zeitmessung für GridSearch (Training + Tuning) starten
    start_time = time.time()
    
    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=5,
        scoring='f1_macro', # angepasst auf f1_macro wie bei deinem Cross-Val Block
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    best_models[name] = grid_search.best_estimator_
    
    # Metriken extrahieren
    best_score = grid_search.best_score_
    test_score = grid_search.score(X_test, y_test)
    best_params_str = str(grid_search.best_params_).replace("{", "").replace("}", "").replace("'", "")

    # HTML-Baustein für jedes Modell generieren
    results_html += f"""
    <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:10px; padding:16px; margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <h4 style="margin:0; color:#1e293b; font-size:18px;">{name}</h4>
            <span style="color:#64748b; font-size:13px;">Dauer: <b>{elapsed_time:.2f} Sek.</b></span>
        </div>
        <div style="font-size:13px; color:#475569; margin-bottom:12px;">
            <b>Beste Parameter:</b> {best_params_str}
        </div>
        <div style="display:flex; gap:16px;">
            <div style="background:#eff6ff; border-radius:8px; padding:12px; flex:1;">
                <div style="color:#475569; font-size:12px;">Bester CV-Score (F1 Macro)</div>
                <div style="font-size:22px; font-weight:700; color:#1d4ed8;">{best_score:.3f}</div>
            </div>
            <div style="background:#f0fdf4; border-radius:8px; padding:12px; flex:1;">
                <div style="color:#475569; font-size:12px;">Test-Score (F1 Macro)</div>
                <div style="font-size:22px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>
    </div>
    """

# Gesamtausgabe in Jupyter rendern
display(HTML(f"""
<div style="font-family:Arial, sans-serif; max-width:800px; margin:16px 0;">
    <h3 style="margin-bottom:12px;">Hyperparameter-Tuning & Grid Search Ergebnisse</h3>
    {results_html}
</div>
"""))

In [ ]:
pipelines = {
    "Logistic Regression": pipe_logreg,
    "K-Nearest Neighbors": pipe_knn,
    "Random Forest": pipe_rf,
    "Hist Gradient Boosting": pipe_hgb
}

for name, pipe in pipelines.items():
    # Zeitmessung starten
    start_time = time.time()
    
    # Cross-Validation mit f1_macro
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1_macro", n_jobs=-1)
    
    # Zeitmessung beenden
    elapsed_time = time.time() - start_time
    
    mean_score = scores.mean()
    std_score = scores.std()

    folds_html = "".join(
        f"""
        <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px;
                    padding:12px; text-align:center;">
            <div style="color:#64748b; font-size:13px;">Fold {i}</div>
            <div style="font-size:22px; font-weight:700; color:#2563eb;">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(scores, start=1)
    )

    display(HTML(f"""
    <div style="font-family:Arial, sans-serif; max-width:800px; margin:16px 0;">
        <h3 style="margin-bottom:8px;">Cross-Validation (F1 Macro): {name}</h3>
        <div style="color:#64748b; font-size:13px; margin-bottom:12px;">Rechenzeit: {elapsed_time:.2f} Sekunden</div>

        <div style="display:flex; gap:16px; margin-bottom:16px;">
            <div style="background:#eff6ff; border-radius:10px; padding:16px; min-width:160px;">
                <div style="color:#475569;">Durchschnitt</div>
                <div style="font-size:30px; font-weight:700; color:#1d4ed8;">{mean_score:.3f}</div>
            </div>

            <div style="background:#f8fafc; border-radius:10px; padding:16px; min-width:160px;">
                <div style="color:#475569;">Standardabweichung</div>
                <div style="font-size:30px; font-weight:700; color:#334155;">{std_score:.3f}</div>
            </div>
        </div>

        <div style="display:grid; grid-template-columns:repeat(5, 1fr); gap:10px;">
            {folds_html}
        </div>
    </div>
    """))

    

## Hyperparameter tunning

## Grid Search

## Random Search

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print('Beste Parameter:', grid_search.best_params_)
print(f'Bester F1-Macro-Score: {grid_search.best_score_:.3f}')

best_grid_model = grid_search.best_estimator_

In [ ]:
param_distributions = {
    'classifier__n_estimators': randint(100, 500),
    'classifier__max_depth': [None, 5, 10, 20, 30],
    'classifier__min_samples_split': randint(2, 15),
    'classifier__min_samples_leaf': randint(1, 8),
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X, y)

print('Beste Parameter:', random_search.best_params_)
print(f'Bester F1-Macro-Score: {random_search.best_score_:.3f}')

best_random_model = random_search.best_estimator_
